# Conurbação Urbana — Litoral SC / Blumenau
### Pipeline unificado: Segmentação → Clusters → Predição

**Fluxo:** todas as imagens são carregadas uma única vez na célula 2 e reutilizadas em todo o notebook.

---
## Célula 1 — Imports

In [ ]:
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from skimage import io
from skimage.color import rgb2gray
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import euclidean_distances
from scipy.spatial.distance import cdist

---
## Célula 2 — Carregamento único de todas as imagens

> **Por que aqui?** Antes, `carregar_imagem()` era chamada repetidamente em cada análise,
> relendo o disco 4–5 vezes por ano. Aqui carregamos tudo uma vez e mantemos em memória.
> As variáveis `anos` e `imagens` alimentam todas as células seguintes.

In [ ]:
def carregar_imagem(nome):
    img = io.imread(nome)
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = img[:, :, :3]
        img = rgb2gray(img)
    return (img * 255).astype(np.uint8)


anos    = list(range(2015, 2026))
imagens = [carregar_imagem(f'assets/Blumenau{ano}.png') for ano in anos]

# Garante resolução uniforme (usa a de 2015 como referência)
h_ref, w_ref = imagens[0].shape
imagens = [
    cv2.resize(img, (w_ref, h_ref)) if img.shape != (h_ref, w_ref) else img
    for img in imagens
]

print(f"Imagens carregadas: {len(imagens)} anos ({anos[0]}–{anos[-1]})")
print(f"Resolução de referência: {h_ref}×{w_ref} px")

---
## Célula 3 — Segmentação em camadas (2015 e 2025 lado a lado)

> Mostra as três camadas de intensidade para os dois extremos temporais do dataset.
> Usa `imagens[0]` (2015) e `imagens[-1]` (2025) — sem reler o disco.

In [ ]:
def segmentar(img):
    """Retorna as três camadas de intensidade de uma imagem já carregada."""
    return {
        'c1': (img >= 0)   & (img < 50),
        'c2': (img >= 50)  & (img < 150),
        'c3': (img >= 150) & (img <= 255),
    }


img_2015 = imagens[0]
img_2025 = imagens[-1]
seg_2015 = segmentar(img_2015)
seg_2025 = segmentar(img_2025)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, (ano, img, seg) in enumerate([
    (2015, img_2015, seg_2015),
    (2025, img_2025, seg_2025),
]):
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original {ano}')

    for col, (nome, chave) in enumerate([('Escuros (C1)', 'c1'),
                                          ('Médios (C2)',  'c2'),
                                          ('Claros (C3)',  'c3')], start=1):
        axes[row, col].imshow(seg[chave], cmap='gray')
        axes[row, col].set_title(f'{nome} — {ano}')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Segmentação por intensidade — 2015 vs 2025', fontsize=13)
plt.tight_layout()
plt.show()

---
## Célula 4 — Evolução temporal da camada 2 (grid de todos os anos)

> Mesmo resultado da célula original, mas sem reler imagens — usa o `imagens` da Célula 2.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, (ano, img) in enumerate(zip(anos, imagens)):
    camada2 = (img >= 50) & (img < 150)
    axes[i].imshow(camada2, cmap='gray')
    axes[i].set_title(str(ano))
    axes[i].axis('off')

axes[-1].axis('off')
plt.suptitle('Camada 2 (franja urbana) — evolução 2015–2025', fontsize=13)
plt.tight_layout()
plt.show()

---
## Célula 5 — Área da camada 2 ao longo dos anos

> Quantifica o crescimento da franja urbana. Também computa crescimento % das três camadas
> entre 2015 e 2025 — consolida o que antes eram duas células separadas.

In [ ]:
areas_c1, areas_c2, areas_c3 = [], [], []

for img in imagens:
    seg = segmentar(img)
    areas_c1.append(int(np.sum(seg['c1'])))
    areas_c2.append(int(np.sum(seg['c2'])))
    areas_c3.append(int(np.sum(seg['c3'])))

# ── Gráfico de área da C2 ──────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(anos, areas_c2, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Ano')
plt.ylabel('Área da camada 2 (pixels)')
plt.title('Expansão da franja urbana (camada 2) — 2015–2025')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── Crescimento % 2015 → 2025 ──────────────────────────────────────────────
def crescimento_pct(a_ini, a_fim):
    return ((a_fim - a_ini) / a_ini) * 100 if a_ini > 0 else float('nan')

print(f"Crescimento camada 1 (escuros): {crescimento_pct(areas_c1[0], areas_c1[-1]):+.1f}%")
print(f"Crescimento camada 2 (médios):  {crescimento_pct(areas_c2[0], areas_c2[-1]):+.1f}%")
print(f"Crescimento camada 3 (claros):  {crescimento_pct(areas_c3[0], areas_c3[-1]):+.1f}%")

---
## Célula 6 — Diferença de intensidade 2025 − 2015

> Mapa de calor seismic: vermelho = ficou mais claro (mais luz), azul = ficou mais escuro.

In [ ]:
diferenca = img_2025.astype(np.int16) - img_2015.astype(np.int16)

plt.figure(figsize=(10, 8))
plt.imshow(diferenca, cmap='seismic', vmin=-128, vmax=128)
plt.colorbar(label='Variação de intensidade (px)')
plt.title('Diferença de luminosidade 2025 − 2015')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Célula 7 — Clusters DBSCAN com cores (evolução 2015–2025)

> Usa sua implementação mais recente: morfologia de abertura, limiar até 210,
> filtro de tamanho mínimo de 600 px, paleta de 30 cores, legenda por subplot.
>
> **Integração com o pipeline:** usa a lista `imagens` carregada na Célula 2 —
> nenhuma imagem é relida. O resultado `clusters_por_ano` é usado pela Célula 8
> (distância entre clusters) e pode alimentar qualquer análise futura.

In [ ]:
CORES = [
    [52,  152, 219], [231, 76,  60],  [46,  204, 113], [241, 196, 15],
    [155, 89,  182], [230, 126, 34],  [26,  188, 156], [236, 240, 241],
    [22,  160, 133], [192, 57,  43],  [142, 68,  173], [39,  174, 96],
    [41,  128, 185], [243, 156, 18],  [189, 195, 199], [127, 140, 141],
    [44,  62,  80],  [211, 84,  0],   [39,  174, 96],  [93,  173, 226],
    [250, 219, 216], [169, 223, 191], [250, 215, 160], [215, 189, 226],
    [174, 214, 241], [169, 204, 227], [100, 181, 246], [255, 138, 101],
    [102, 187, 106], [255, 241, 118],
]

kernel = np.ones((1, 1), np.uint8)
clusters_por_ano = {}

for i, ano in enumerate(anos):
    img_limpa = cv2.morphologyEx(imagens[i], cv2.MORPH_OPEN, kernel, iterations=2)
    mascara   = (img_limpa >= 50) & (img_limpa < 210)
    coords    = np.column_stack(np.where(mascara))

    if len(coords) == 0:
        print(f"{ano}: sem pixels")
        continue

    labels   = DBSCAN(eps=8, min_samples=30).fit_predict(coords)
    clusters = [
        coords[labels == lbl]
        for lbl in set(labels)
        if lbl != -1 and (labels == lbl).sum() >= 600
    ]
    clusters = sorted(clusters, key=len, reverse=True)
    clusters_por_ano[ano] = clusters
    print(f"{ano}: {len(clusters)} clusters | tamanhos: {[len(c) for c in clusters]}")


# ── Visualização: grid de todos os anos ───────────────────────────────────────
anos_disp = sorted(clusters_por_ano.keys())
n_anos    = len(anos_disp)
cols, rows = 4, math.ceil(n_anos / 4)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 7))
axes = axes.flatten()

for idx, ano in enumerate(anos_disp):
    clusters = clusters_por_ano[ano]
    img_ref  = imagens[anos.index(ano)]
    mapa     = np.zeros((*img_ref.shape, 3), dtype=np.uint8)

    for ci, cluster in enumerate(clusters):
        mapa[cluster[:, 0], cluster[:, 1]] = CORES[ci % len(CORES)]

    axes[idx].imshow(mapa)
    axes[idx].set_title(f'{ano} — {len(clusters)} clusters', fontsize=11)
    axes[idx].axis('off')

    legenda = [
        Patch(color=[c / 255 for c in CORES[ci % len(CORES)]],
              label=f'C{ci+1}: {len(clusters[ci]):,}px')
        for ci in range(len(clusters))
    ]
    axes[idx].legend(handles=legenda, loc='lower left', fontsize=7,
                     framealpha=0.8, facecolor='white', ncol=2)

for j in range(n_anos, len(axes)):
    axes[j].axis('off')

plt.suptitle('Evolução dos clusters urbanos — 2015–2025', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## Célula 8 — Distância mínima entre os dois maiores clusters ao longo do tempo

> **Integração com Célula 7:** usa `clusters_por_ano` já calculado — sem reprocessar DBSCAN.
> Também calcula e plota o centroide da camada 2 para 2015 e 2025.

In [ ]:
anos_dist, distancias = [], []

for ano in anos_disp:
    clusters = clusters_por_ano[ano]
    if len(clusters) < 2:
        continue
    dist = euclidean_distances(clusters[0], clusters[1]).min()
    anos_dist.append(ano)
    distancias.append(dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Gráfico de distância ───────────────────────────────────────────────────
axes[0].plot(anos_dist, distancias, marker='o', color='crimson', linewidth=2)
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Distância mínima (pixels)')
axes[0].set_title('Convergência dos dois maiores clusters')
axes[0].grid(alpha=0.3)

# ── Deslocamento do centroide 2015 → 2025 ─────────────────────────────────
coords_2015 = np.column_stack(np.where((imagens[0]  >= 50) & (imagens[0]  < 150)))
coords_2025 = np.column_stack(np.where((imagens[-1] >= 50) & (imagens[-1] < 150)))

c2015 = coords_2015.mean(axis=0)  # (y, x)
c2025 = coords_2025.mean(axis=0)

axes[1].imshow(imagens[-1], cmap='gray')
axes[1].scatter(c2015[1], c2015[0], c='red',  s=180, zorder=5, label='Centroide 2015')
axes[1].scatter(c2025[1], c2025[0], c='cyan', s=180, zorder=5, label='Centroide 2025')
axes[1].annotate('', xy=(c2025[1], c2025[0]), xytext=(c2015[1], c2015[0]),
                 arrowprops=dict(arrowstyle='->', color='yellow', lw=2))
axes[1].legend()
axes[1].set_title('Deslocamento do centroide urbano 2015 → 2025')
axes[1].axis('off')

plt.tight_layout()
plt.show()

deslocamento = np.linalg.norm(c2025 - c2015)
print(f"Centroide 2015: y={c2015[0]:.1f}, x={c2015[1]:.1f}")
print(f"Centroide 2025: y={c2025[0]:.1f}, x={c2025[1]:.1f}")
print(f"Magnitude do deslocamento: {deslocamento:.1f} px")
if len(distancias) >= 2:
    print(f"Distância 2015: {distancias[0]:.1f} px  →  Distância {anos_dist[-1]}: {distancias[-1]:.1f} px")

---
## Célula 9 — Predição por regressão linear pixel a pixel (2035)

> Usa sua função `prever_expansao_pixel` sem alterações.
> **Integração com o pipeline:** recebe diretamente `imagens` e `anos` da Célula 2.
> Também corrige o bug de `novos_pixels` (variável indefinida no print final).

In [ ]:
def prever_expansao_pixel(imagens_por_ano, anos, ano_alvo=2035):
    """
    Regressão linear por pixel usando álgebra matricial (einsum).
    Para cada pixel (h, w) ajusta y = a*t + b sobre a série temporal
    e extrapola para ano_alvo.
    """
    anos_arr  = np.array(anos, dtype=np.float32)
    ano_alvo  = float(ano_alvo)
    stack     = np.stack(imagens_por_ano, axis=0).astype(np.float32)  # (T, H, W)

    x_mean    = anos_arr.mean()
    y_mean    = stack.mean(axis=0)                                     # (H, W)
    x_diff    = anos_arr - x_mean                                      # (T,)

    num       = np.einsum('t,thw->hw', x_diff, stack - y_mean)        # inclinação numerador
    denom     = (x_diff ** 2).sum()                                    # escalar

    a         = num / denom                                            # (H, W) inclinação
    b         = y_mean - a * x_mean                                    # (H, W) intercepto

    mapa      = a * ano_alvo + b
    mapa[stack.max(axis=0) <= 5] = 0                                   # zera pixels sempre escuros

    return np.clip(mapa, 0, 255).astype(np.uint8)


print("Calculando previsão para 2035...")
mapa_2035 = prever_expansao_pixel(imagens, anos, ano_alvo=2035)
print("Concluído!")

camada2_2025    = (imagens[-1] >= 50) & (imagens[-1] < 150)
camada2_prevista = (mapa_2035  >= 50) & (mapa_2035  < 150)
novos_pixels     = camada2_prevista & ~camada2_2025        # área de expansão líquida

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(imagens[-1], cmap='gray')
axes[0].set_title('Luzes reais — 2025')
axes[0].axis('off')

axes[1].imshow(camada2_prevista, cmap='hot')
axes[1].set_title('Previsão camada 2 — 2035')
axes[1].axis('off')

comparacao = np.zeros((*camada2_2025.shape, 3), dtype=np.uint8)
comparacao[camada2_2025]   = [100, 100, 100]  # área atual: cinza
comparacao[novos_pixels]   = [255, 80,  0]    # expansão prevista: laranja
axes[2].imshow(comparacao)
axes[2].set_title('Expansão prevista 2025 → 2035\n(cinza = atual, laranja = novo)')
axes[2].axis('off')

plt.suptitle('Previsão de expansão urbana — camada 2', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nÁrea camada 2 em 2025:       {camada2_2025.sum():,} pixels")
print(f"Área camada 2 prevista 2035: {camada2_prevista.sum():,} pixels")
print(f"Expansão prevista:           {novos_pixels.sum():,} pixels novos")

---
## Célula 10 — Previsão da camada 3 (núcleo consolidado) em 2035

> Reutiliza `mapa_2035` calculado na Célula 9 — sem recalcular a regressão.

In [ ]:
camada3_2025    = imagens[-1] >= 150
camada3_prevista = mapa_2035  >= 150
novos_c3        = camada3_prevista & ~camada3_2025

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(imagens[-1], cmap='gray')
axes[0].set_title('Luzes reais — 2025')
axes[0].axis('off')

axes[1].imshow(camada3_prevista, cmap='hot')
axes[1].set_title('Previsão camada 3 — 2035')
axes[1].axis('off')

comparacao = np.zeros((*camada3_2025.shape, 3), dtype=np.uint8)
comparacao[camada3_2025] = [100, 100, 100]
comparacao[novos_c3]     = [255, 80,  0]
axes[2].imshow(comparacao)
axes[2].set_title('Expansão prevista 2025 → 2035\n(cinza = atual, laranja = novo)')
axes[2].axis('off')

plt.suptitle('Previsão de expansão urbana — camada 3 (núcleo consolidado)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nÁrea camada 3 em 2025:       {camada3_2025.sum():,} pixels")
print(f"Área camada 3 prevista 2035: {camada3_prevista.sum():,} pixels")
print(f"Expansão prevista:           {novos_c3.sum():,} pixels novos")

---
## Célula 11 — Clusters sobrepostos à previsão 2035

> **Integração clusters ↔ predição:** aplica o mesmo DBSCAN da Célula 7
> sobre o `mapa_2035` para mostrar quantos e quais clusters sobrevivem
> (ou se fundem) na projeção — evidência visual da conurbação prevista.

In [ ]:
# Reutiliza os mesmos parâmetros DBSCAN da Célula 7 para comparabilidade
mascara_prev = (mapa_2035 >= 50) & (mapa_2035 < 210)
coords_prev  = np.column_stack(np.where(mascara_prev))

labels_prev  = DBSCAN(eps=8, min_samples=30).fit_predict(coords_prev)
clusters_prev = [
    coords_prev[labels_prev == lbl]
    for lbl in set(labels_prev)
    if lbl != -1 and (labels_prev == lbl).sum() >= 600
]
clusters_prev = sorted(clusters_prev, key=len, reverse=True)

mapa_vis = np.zeros((*mapa_2035.shape, 3), dtype=np.uint8)
for ci, cluster in enumerate(clusters_prev):
    mapa_vis[cluster[:, 0], cluster[:, 1]] = CORES[ci % len(CORES)]

# Comparação: clusters 2025 vs clusters 2035 previstos
clusters_2025 = clusters_por_ano.get(2025, [])
mapa_2025_vis = np.zeros((*imagens[-1].shape, 3), dtype=np.uint8)
for ci, cluster in enumerate(clusters_2025):
    mapa_2025_vis[cluster[:, 0], cluster[:, 1]] = CORES[ci % len(CORES)]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(mapa_2025_vis)
axes[0].set_title(f'Clusters reais 2025 ({len(clusters_2025)} clusters)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(mapa_vis)
axes[1].set_title(f'Clusters previstos 2035 ({len(clusters_prev)} clusters)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Fusão de clusters: evidência de conurbação prevista', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Clusters em 2025: {len(clusters_2025)}")
print(f"Clusters previstos em 2035: {len(clusters_prev)}")
if len(clusters_2025) > len(clusters_prev):
    print("→ Redução no número de clusters: clusters se fundiram (conurbação em curso)")
elif len(clusters_2025) == len(clusters_prev):
    print("→ Número estável de clusters: crescimento sem fusão no período projetado")